In [1]:
import pandas as pd

# Load only the header
file_path = '~/Documents/Research/diagnosis_data_with_ccsr.csv'
df = pd.read_csv(file_path, nrows=0)

# Show column names
print(df.columns.tolist())


['Age', 'Female', 'K088', 'Pay1', 'NCHS', 'Total Cost', 'ZIP', 'Diagnosis', 'CCSR Category']


In [2]:
import pandas as pd

# Path to your file
file_path = '~/Documents/Research/diagnosis_data_with_ccsr.csv'

# Read the first 5 rows
df = pd.read_csv(file_path, nrows=5)

# Display the data
print(df)


   Age  Female     K088  Pay1  NCHS  Total Cost  ZIP Diagnosis CCSR Category
0    4       0     J069     2     1       914.0    3       RSP        RSP006
1   69       0    I4891     1     5      2868.0    4       CIR        CIR017
2   49       0  S9032XA     2     1      1274.0    4       INJ        INJ017
3   67       0    K1370     1     1       840.0    3       DEN        DEN002
4   11       1     B349     2     1       987.0    3       INF        INF008


In [3]:
import os
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [4]:
file_path = os.path.expanduser("~/Documents/Research/diagnosis_data_with_ccsr.csv")

chunk_size = 100000
n_rows = 1000000
data_list = []
rows_read = 0

In [5]:
for chunk in pd.read_csv(file_path, chunksize=chunk_size):
    chunk = chunk[(chunk['Total Cost'] >= 0) & (chunk['Age'] >= 0) &
                  (chunk['ZIP'] >= 0) & (chunk['NCHS'] >= 0) & (chunk['Pay1'] >= 0)]
    data_list.append(chunk)
    rows_read += len(chunk)
    if rows_read >= n_rows:
        break

In [6]:
data = pd.concat(data_list, ignore_index=True)

# Remove outliers using IQR for Age and Total Cost
for col in ['Total Cost', 'Age']:
    Q1 = data[col].quantile(0.25)
    Q3 = data[col].quantile(0.75)
    IQR = Q3 - Q1
    data = data[(data[col] >= Q1 - 1.5 * IQR) & (data[col] <= Q3 + 1.5 * IQR)]

In [7]:
features = ['Age', 'Female', 'ZIP', 'NCHS', 'Pay1', 'CCSR Category']
X = data[features]
y = data['Total Cost']


In [8]:
categorical_cols = ['ZIP', 'NCHS', 'Pay1', 'CCSR Category']
X_encoded = pd.get_dummies(X, columns=categorical_cols)

In [9]:
X_train, X_test, y_train, y_test = train_test_split(X_encoded, y, test_size=0.2, random_state=42)

# Train XGBoost model
model = XGBRegressor(n_estimators=100, learning_rate=0.1, max_depth=6, random_state=42)
model.fit(X_train, y_train)


XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=None, device=None, early_stopping_rounds=None,
             enable_categorical=False, eval_metric=None, feature_types=None,
             feature_weights=None, gamma=None, grow_policy=None,
             importance_type=None, interaction_constraints=None,
             learning_rate=0.1, max_bin=None, max_cat_threshold=None,
             max_cat_to_onehot=None, max_delta_step=None, max_depth=6,
             max_leaves=None, min_child_weight=None, missing=nan,
             monotone_constraints=None, multi_strategy=None, n_estimators=100,
             n_jobs=None, num_parallel_tree=None, ...)

In [10]:
y_pred = model.predict(X_test)
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print(f"MAE: {mae:.2f}")
print(f"RMSE: {rmse:.2f}")
print(f"R² Score: {r2:.4f}")

MAE: 1225.08
RMSE: 1624.59
R² Score: 0.2213
